# 2. Training

One notebook for the three kinds of runs (`METHOD`):

| `METHOD` | network | recipe | script |
|---|---|---|---|
| `ndt` | official NDT-Transformer (`third_party/NDT-Transformer`) | official: lazy quadruplet loss, tuples | `scripts/train_ndt_kitti.py` |
| `bev` | official BEVPlace++ (REIN), or `arch="plain"` | BEVPlace++: triplet loss, hard mining from a descriptor cache | `scripts/train_bev_kitti.py` |
| `myndt` | own NDT network (`src/lpr/models/ndt_net.py`) | the BEVPlace++ recipe, unchanged | `scripts/train_myndt_kitti.py` |

`TRAIN_PARAMS` holds only what differs from the defaults of the method in `src/lpr/experiments.py`
(`TRAIN_DEFAULTS`, `BEV_TRAIN_DEFAULTS`, `MYNDT_TRAIN_DEFAULTS`). A run is stored in `MyDrive/lpr/runs/<method>/<run>/`
(`config.json` with every parameter and the code version, `metrics.csv`, a checkpoint per epoch); its name ends with
the hash of the parameters and the variant, so the same settings always find the same run: an unfinished run
continues after a disconnect, a finished one is not trained again (change `seed` to repeat it).
The settings of every run of the report are in the README.

The data must be on Drive first (notebook 1). Runtime: GPU (T4 for `ndt` and `bev`, L4 / A100 for `myndt` with
8 rotations). Set cell 1, then **Runtime -> Run all**. Evaluation: notebook 3.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
CODE_ON_DRIVE = ""                      # or the path of this code folder copied to Drive (then REPO_URL is not used)
CODE = "/content/lpr"                   # the code on this machine
DRIVE = "/content/drive/MyDrive/lpr"    # everything that is kept: variants, runs, evaluations, tables
KITTI = "/content/kitti"                # poses (and raw scans when needed)

METHOD = "myndt"        # "ndt" | "bev" | "myndt"
MODE = "train"          # "train": train (or continue) the run; "smoke": 30 iterations, nothing kept (time, GPU memory)
SEQ = "00"              # training sequence (frames 0-2999)
VARIANT_TAG = ""        # the data variant (as in notebook 1): only what differs from NDT_DEFAULTS / BEV_DEFAULTS
VARIANT_PARAMS = dict() # e.g. dict(ground="none") with VARIANT_TAG = "ground_none"
RUN_TAG = ""            # name for the run ("" = date + hash)
TRAIN_PARAMS = dict(epochs=20, lr=1e-5, rotations=8, head="loupe", feat_dim=1028, rot_aug=False)   # NDT-REIN

## 2. Drive, code, poses

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
import os, sys, json, glob, subprocess, importlib
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU"
if not os.path.exists(CODE):
    if CODE_ON_DRIVE:
        subprocess.run(["cp", "-r", CODE_ON_DRIVE, CODE], check=True)
    else:
        url = REPO_URL
        try:                            # a private repository: a token in Colab Secrets as GITHUB_TOKEN
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, CODE], check=True)
elif os.path.isdir(f"{CODE}/.git"):
    subprocess.run(["git", "-C", CODE, "pull", "-q"])
%cd {CODE}
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt
sys.path.insert(0, f"{CODE}/src")
import lpr.experiments as ex, lpr.evals as ev, lpr.colab as cl
for m in (ex, ev, cl):
    importlib.reload(m)
try:                                    # container SAS URL (read + list) in Colab Secrets; never in a notebook
    SAS = userdata.get("AZURE_SAS")
except Exception:
    SAS = None
    print("no AZURE_SAS in Colab Secrets: only data that is already on Drive is used")
missing_poses = [s for s in ev.ALL_SEQS if not os.path.exists(f"{KITTI}/poses/{s}.txt")]
if missing_poses:
    !python scripts/download_kitti.py --out {KITTI} --seqs {" ".join(missing_poses)} --poses_only

## 3. Variant and run

In [ ]:
assert METHOD in ("ndt", "bev", "myndt") and MODE in ("train", "smoke")
DATA = "bev" if METHOD == "bev" else "ndt"          # myndt uses the NDT variants
VARIANT, VFULL, v_exists = ex.resolve_variant(DRIVE, DATA, VARIANT_PARAMS, VARIANT_TAG)
RUN, T, run_state = ex.resolve_run(DRIVE, dict(TRAIN_PARAMS, seq=SEQ), VARIANT, RUN_TAG, method=METHOD)
RUN_DIR = f"{DRIVE}/runs/{METHOD}/{RUN}"
LOCAL = f"/content/{DATA}/{VARIANT}"
print("variant:", VARIANT, "" if v_exists else "(not on Drive: notebook 1 first)")
print("run:    ", RUN, f"({run_state})")
print(json.dumps(T))

if METHOD == "ndt":
    train_args = (f"--ndt_dir {LOCAL} --kitti_root {KITTI} --train_pickle {LOCAL}/train_{SEQ}.pickle "
                  f"--ndt_variant {VARIANT} --seq {SEQ} --epochs {T['epochs']} --lr {T['lr']} "
                  f"--milestones {' '.join(map(str, T['milestones']))} --batch_queries {T['batch_queries']} "
                  f"--positives {T['positives']} --negatives {T['negatives']} --margin1 {T['margin1']} "
                  f"--margin2 {T['margin2']} --hard_neg_after_epoch {T['hard_neg_after_epoch']} --seed {T['seed']}"
                  + (" --amp" if T["amp"] else "") + (" --cov_transform" if T["cov_transform"] else "")
                  + (" --rot_aug" if T["rot_aug"] else "") + (f" --rot_points {T['rot_points']}" if T["rot_points"] else ""))
elif METHOD == "bev":
    train_args = (f"--bev_dir {LOCAL} --kitti_root {KITTI} --bev_variant {VARIANT} --seq {T['seq']} "
                  f"--train_frames {T['train_frames']} --pos_thr {T['pos_thr']} --neg_thr {T['neg_thr']} "
                  f"--epochs {T['epochs']} --lr {T['lr']} --batch {T['batch']} --negatives {T['negatives']} "
                  f"--margin {T['margin']} --seed {T['seed']}" + ("" if T["rot_aug"] else " --no_rot_aug")
                  + (f" --arch {T['arch']}" if T["arch"] != "rein" else "")
                  + (f" --hard_neg_after_epoch {T['hard_neg_after_epoch']}" if T["hard_neg_after_epoch"] else ""))
else:
    train_args = (f"--ndt_dir {LOCAL} --kitti_root {KITTI} --ndt_variant {VARIANT} "
                  + " ".join(f"--{k} {T[k]}" for k in ("seq", "train_frames", "pos_thr", "neg_thr", "epochs", "lr", "batch",
                                                      "negatives", "margin", "seed", "rot_points", "rotations", "head", "feat_dim"))
                  + " " + " ".join(f"--{k}" if T[k] else f"--no_{k}" for k in ("rot_aug", "amp", "stn", "cov_transform")))
if T.get("beam_aug"):
    train_args += " --beam_aug " + " ".join(map(str, T["beam_aug"]))
SCRIPT = {"ndt": "train_ndt_kitti.py", "bev": "train_bev_kitti.py", "myndt": "train_myndt_kitti.py"}[METHOD]
print(SCRIPT, train_args)

## 4. Training data on this machine
The training sequence of the variant (Drive, or the blob container), the point-rotated copies for `rot_points=N`
and the 32 / 16-beam variants for `beam_aug`. NDT-Transformer also needs its training tuples (frames 0-2999,
positive < 5 m, negative > 7 m, official pickle format); the BEVPlace++ recipe finds them from the poses itself.

In [ ]:
LOCAL, have = cl.to_machine(DRIVE, DATA, VARIANT, [SEQ], SAS)
if SEQ not in have:
    raise SystemExit(f"sequence {SEQ} of {VARIANT} is not on Drive: notebook 1 first")
if DATA == "ndt" and T.get("rot_points"):
    sub, tar = ex.rot_dir_name(SEQ, T["rot_points"]), f"{DRIVE}/ndt/{VARIANT}/{ex.rot_archive_name(SEQ, T['rot_points'])}"
    if not os.path.exists(tar):
        raise SystemExit(f"{os.path.basename(tar)} is not on Drive: notebook 1 with ROT_COPIES = {T['rot_points']}")
    if not os.path.isdir(f"{LOCAL}/{sub}"):
        !tar -xf {tar} -C {LOCAL} {sub} {sub}_stats.csv
cl.sparse_to_machine(DRIVE, DATA, VARIANT, [b for b in T.get("beam_aug", []) if b != 64], [SEQ], SAS)
if METHOD == "ndt":
    !python scripts/make_kitti_tuples.py --kitti_root {KITTI} --seq {SEQ} --ndt_dir {LOCAL} --out {LOCAL}/train_{SEQ}.pickle --pos_thr {T['pos_thr']} --neg_thr {T['neg_thr']}

## 5. Training
`smoke`: seconds per iteration x iterations per epoch (about 1500 for `ndt`, 750 for `bev` and `myndt`) is roughly
one epoch; the log also shows the peak GPU memory. `train`: a checkpoint every few minutes, so a disconnect costs
little: Run all again and the run continues. If a batch does not fit in GPU memory, the tuples go through one by
one with accumulated gradients (`split_forward` in `metrics.csv`).

In [ ]:
if MODE == "smoke":
    !rm -rf /content/smoke
    !python scripts/{SCRIPT} {train_args} --log_dir /content/smoke --epochs 1 --max_iters 30 --eval_every 0 2>&1 | grep -v Warning
elif run_state == "finished":
    print("finished before:", RUN_DIR)
else:
    !python scripts/{SCRIPT} {train_args} --log_dir {RUN_DIR} 2>&1 | grep -v Warning

## 6. Training curve (the recall during training is on the training sequence: monitoring only)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
if os.path.exists(f"{RUN_DIR}/metrics.csv"):
    m = pd.read_csv(f"{RUN_DIR}/metrics.csv").drop_duplicates("epoch", keep="last")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(m.epoch, m.train_loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
    if "R@1" in m:
        ax[1].plot(m.epoch, m["R@1"], marker="o", label="R@1"); ax[1].plot(m.epoch, m["R@1%"], marker="o", label="R@1%")
        ax[1].legend()
    ax[1].set_title(f"KITTI {SEQ} recall (%), {RUN}"); ax[1].set_xlabel("epoch")
    plt.show()